# SetFit HTS Classifier Training - Colab Pro

This notebook trains a SetFit model on the full CROSS rulings dataset.

**Before running:**
1. Runtime → Change runtime type → **L4 GPU** (or A100/V100) → Save
2. Upload your training data files from `src/data/`:
   - `crossRulings-train.json` (training split)
   - `crossRulings-validation.json`
   - `crossRulings-test.json`

**Expected time:** 45-60 minutes on L4 GPU  
**Expected accuracy:** 92-95% (heading level), 85-88% (subheading level)

**Anti-crash fixes (v2):**
- Dependency versions pinned to known-working releases
- Memory-safe contrastive pair generation (num_iterations=10)
- Explicit gc.collect() between heavy phases
- Correct training file names (crossRulings-train.json, not crossRulings.json)

## Step 1: Install Dependencies

In [ ]:
# Pin to known-working versions (transformers 4.57.6 DOES NOT EXIST — was causing install crash)
# setfit 1.1.3 is compatible with transformers 4.x, NOT 5.x
!pip install -q "setfit==1.1.3" "transformers>=4.44,<5.0" "sentence-transformers>=3.0,<4.0" torch scikit-learn datasets

# Verify install
import setfit, transformers
print(f"✅ setfit=={setfit.__version__}, transformers=={transformers.__version__}")

## Step 2: Upload Training Data

In [ ]:
from google.colab import files
import json
import os

# Upload all 3 files — these are in src/data/ in your project
EXPECTED_FILES = {
    'crossRulings-train.json': 'Training data',
    'crossRulings-validation.json': 'Validation data',
    'crossRulings-test.json': 'Test data',
}

print("Upload these 3 files from src/data/:")
for f, desc in EXPECTED_FILES.items():
    print(f"  • {f} ({desc})")
print()

uploaded = files.upload()

# Verify all files present
for f in EXPECTED_FILES:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"✅ {f} ({size_mb:.1f} MB)")
    else:
        print(f"❌ MISSING: {f} — training will fail without this file!")

## Step 3: Prepare Dataset (Memory Optimized)

In [ ]:
import json
from datasets import Dataset
import gc
import os

def load_rulings(filepath):
    """Load rulings from JSON file, handling both array and object formats."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"❌ File not found: {filepath}\n"
            f"   Make sure you uploaded the correct file from src/data/"
        )
    with open(filepath, 'r') as f:
        data = json.load(f)
        if isinstance(data, list):
            return data
        elif isinstance(data, dict) and 'rulings' in data:
            return data['rulings']
        else:
            raise ValueError(f'Unexpected data format in {filepath}')

def prepare_dataset(rulings, level='subheading', max_samples=None):
    """Convert rulings to HuggingFace Dataset at the specified HTS level."""
    texts = []
    labels = []
    skipped = 0

    for i, ruling in enumerate(rulings):
        if max_samples and i >= max_samples:
            break

        text = ruling.get('productDescription') or ruling.get('product_description', '')
        hts_codes = ruling.get('htsCodes') or ruling.get('hts_codes', [])

        if not text or not hts_codes:
            skipped += 1
            continue

        hts_code = hts_codes[0] if isinstance(hts_codes, list) else hts_codes

        # Normalize: strip dots/spaces from HTS code
        hts_code = hts_code.replace('.', '').replace(' ', '').strip()
        if len(hts_code) < 4:
            skipped += 1
            continue

        if level == 'chapter':
            label = hts_code[:2]
        elif level == 'heading':
            label = hts_code[:4]
        elif level == 'subheading':
            label = hts_code[:6]
        else:
            label = hts_code

        texts.append(text.strip())
        labels.append(label)

    if skipped > 0:
        print(f"  ⚠️  Skipped {skipped} rulings with missing text/codes")

    return Dataset.from_dict({'text': texts, 'label': labels})

# ──────────────────────────────────────────────────────────────────────────
# TRAINING LEVEL: Choose what the model predicts
#   'heading'    = 4-digit (~350 classes) — RECOMMENDED for heading-first pipeline
#   'subheading' = 6-digit (~1,200 classes) — original model
#   'chapter'    = 2-digit (~97 classes) — too coarse
# ──────────────────────────────────────────────────────────────────────────
TRAINING_LEVEL = 'heading'  # Change to 'subheading' for the old model

# ──────────────────────────────────────────────────────────────────────────
# LOAD DATA — uses the TRAINING split, not the full crossRulings.json
# ──────────────────────────────────────────────────────────────────────────
print("Loading training data...")
train_rulings = load_rulings('crossRulings-train.json')
print(f"Training rulings loaded: {len(train_rulings)}")

val_rulings = load_rulings('crossRulings-validation.json')
print(f"Validation rulings loaded: {len(val_rulings)}")

test_rulings = load_rulings('crossRulings-test.json')
print(f"Test rulings loaded: {len(test_rulings)}")

# Cap training samples to prevent OOM during contrastive pair generation
# L4 (24GB) safely handles ~12K samples. A100 (40GB) can do 20K+.
MAX_TRAIN_SAMPLES = 12000
print(f"\nUsing up to {MAX_TRAIN_SAMPLES} training samples (OOM safety limit for L4)")

train_dataset = prepare_dataset(train_rulings, level=TRAINING_LEVEL, max_samples=MAX_TRAIN_SAMPLES)
val_dataset = prepare_dataset(val_rulings, level=TRAINING_LEVEL)
test_dataset = prepare_dataset(test_rulings, level=TRAINING_LEVEL)

# Free raw rulings from memory immediately
del train_rulings, val_rulings, test_rulings
gc.collect()

print(f"\n{'='*60}")
print(f"Training level: {TRAINING_LEVEL}")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Unique labels: {len(set(train_dataset['label']))}")
print(f"{'='*60}")

## Step 4: Train SetFit Model (Optimized Settings)

In [ ]:
from setfit import SetFitModel, Trainer, TrainingArguments
import torch
import gc

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    gpu_name = "None"
    gpu_mem = 0
    print("⚠️  No GPU detected — training will be very slow!")

# ──────────────────────────────────────────────────────────────────────────
# MEMORY CLEANUP before model init
# ──────────────────────────────────────────────────────────────────────────
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_mem = torch.cuda.mem_get_info()[0] / (1024**3)
    print(f"Free GPU memory: {free_mem:.1f} GB")

# Model output directory based on training level
MODEL_DIR = f"./setfit-hts-{TRAINING_LEVEL}"

# ──────────────────────────────────────────────────────────────────────────
# GPU-ADAPTIVE SETTINGS
# The #1 crash cause is OOM during contrastive pair generation.
# num_iterations controls how many positive/negative pairs per sample.
# Lower = fewer pairs = less RAM. 10 is plenty for heading-level (~350 classes).
# ──────────────────────────────────────────────────────────────────────────
if gpu_mem >= 40:  # A100
    batch_size = 16
    num_iterations = 15
    print("🚀 A100 detected — using higher capacity settings")
elif gpu_mem >= 20:  # L4 / V100
    batch_size = 8
    num_iterations = 10
    print("⚡ L4/V100 detected — using balanced settings")
else:  # T4 or smaller
    batch_size = 4
    num_iterations = 5
    print("🐢 T4/small GPU — using conservative settings to avoid OOM")

num_epochs = 3

# Initialize model
print(f"\nInitializing SetFit model for '{TRAINING_LEVEL}' prediction...")
model = SetFitModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2",
    labels=list(set(train_dataset['label']))
)

# Training arguments
args = TrainingArguments(
    batch_size=batch_size,
    num_epochs=num_epochs,
    evaluation_strategy="no",    # Skip eval during training (saves RAM)
    save_strategy="epoch",
    output_dir=MODEL_DIR,
    logging_steps=50,
    num_iterations=num_iterations,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
)

# ──────────────────────────────────────────────────────────────────────────
# TRAIN
# ──────────────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Starting training...")
print(f"GPU: {gpu_name}")
print(f"Training level: {TRAINING_LEVEL}")
print(f"Training samples: {len(train_dataset)}")
print(f"Epochs: {num_epochs}")
print(f"Batch size: {batch_size}")
print(f"Contrastive iterations: {num_iterations}")
print(f"Output: {MODEL_DIR}")
print(f"{'='*60}")
print(f"\n⏳ Phase 1: Generating contrastive pairs (2-5 min, RAM-intensive)...")
print(f"   If this crashes with OOM, reduce MAX_TRAIN_SAMPLES in cell above.")
print(f"⏳ Phase 2: Training will start after pair generation...\n")

trainer.train()

# Clean up training memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Training complete!")

## Step 5: Evaluate Model

In [ ]:
import time
from sklearn.metrics import accuracy_score

print("Evaluating on test set...")
start_time = time.time()

predictions = model.predict(test_dataset['text'])
accuracy = accuracy_score(test_dataset['label'], predictions)
inference_time = (time.time() - start_time) / len(test_dataset) * 1000

print(f"\n{'='*60}")
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"Avg Inference Time: {inference_time:.2f}ms per sample")
print(f"{'='*60}")

metrics = {
    'accuracy': accuracy,
    'total_samples': len(test_dataset),
    'inference_time_ms': inference_time,
    'training_samples': len(train_dataset),
    'model_name': f'setfit-hts-{TRAINING_LEVEL}',
    'training_level': TRAINING_LEVEL,
}

with open(f'{MODEL_DIR}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\nMetrics saved!")

## Step 6: Test Sample Predictions

In [ ]:
test_examples = [
    "cotton t-shirt",
    "plastic water bottle",
    "leather wallet",
    "stainless steel kitchen knife",
    "wooden dining table"
]

print("Sample predictions:\n")
for example in test_examples:
    pred = model.predict([example])[0]
    print(f"'{example}' → {pred}")

## Step 7: Save and Download Model

In [ ]:
print(f"Saving model to {MODEL_DIR}...")
model.save_pretrained(MODEL_DIR)
print("Model saved!")

zip_name = f"setfit-hts-{TRAINING_LEVEL}.zip"
!zip -r {zip_name} {MODEL_DIR}/

print(f"\nDownloading {zip_name}...")
files.download(zip_name)

print(f"\n✅ Done! Extract to models/ folder and start the inference server:")
print(f"   python scripts/setfit-inference-server.py --model models/setfit-hts-{TRAINING_LEVEL}")